## **Shap Values**

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import shap
import matplotlib.pyplot as plt

df = pd.read_csv('china_used_cars.csv')

# Drop target + columns that duplicate/derive from it
drop_cols = ['price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# --- SHAP: modern Explanation-object API ---
explainer = shap.TreeExplainer(model)
exp = explainer(X_test)   # shap.Explanation, handles base_value shapes internally

# 1. Global summary — which features matter most, and in which direction
plt.figure()
shap.summary_plot(exp.values, X_test, show=False)
plt.savefig('shap_summary.png', bbox_inches='tight')
plt.close()

# 2. Local explanation — why THIS car got THIS predicted price
plt.figure()
shap.plots.waterfall(exp[0], show=False)
plt.savefig('shap_waterfall_row0.png', bbox_inches='tight')
plt.close()

# 3. Dependence — how one feature's effect changes across its value range
plt.figure()
shap.dependence_plot('motor_power_kw', exp.values, X_test, show=False)
plt.savefig('shap_dependence_power.png', bbox_inches='tight')
plt.close()

# Sanity check: base_value + this row's shap values == the model's actual prediction
pred0 = model.predict(X_test.iloc[[0]])[0]
print("Model prediction:      ", pred0)
print("base_value + shap sum:  ", exp.base_values[0] + exp.values[0].sum())

Model prediction:       7547.242500000001
base_value + shap sum:   7547.242500000899


<Figure size 640x480 with 0 Axes>

## **Multioutput Models**

In [4]:
from sklearn.multioutput import MultiOutputRegressor

y_multi = df[['price', 'mileage_per_year']]
X_train, X_test, y_train, y_test = train_test_split(X, y_multi, test_size=0.2, random_state=42)

multi_model = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
)
multi_model.fit(X_train, y_train)

preds = multi_model.predict(X_test)  # shape: (n_samples, 2) -> [price, mileage_per_year]

In [5]:
target_names = y_multi.columns.tolist()  # ['price', 'mileage_per_year']

shap_values_per_target = {}
for i, target in enumerate(target_names):
    single_model = multi_model.estimators_[i]     # the RF trained for this one target
    explainer = shap.TreeExplainer(single_model)
    sv = explainer.shap_values(X_test)
    shap_values_per_target[target] = sv

    plt.figure()
    shap.summary_plot(sv, X_test, show=False)
    plt.title(f"SHAP summary — target: {target}")
    plt.savefig(f'shap_summary_{target}.png', bbox_inches='tight')
    plt.close()